In [12]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261106 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

# COMMAND 

In [13]:
%%writefile STAARpipeline_Gene_Centric_Coding.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Read command-line arguments supplied after: 
args <- commandArgs(trailingOnly = TRUE)

# Input arguments:
#   1. agds_path          = chromosome-specific annotated GDS file
#   2. obj_nullmodel_path = fitted STAAR/SAIGE null model RData
#   3. annotation_path   = annotation catalog RData
#   4. output_path        = directory for gene-level results
#   5. batch              = batch size / maximum genes per batch
#   6. chr                = chromosome to analyze
agds_path          <- args[1]
obj_nullmodel_path <- args[2]
annotation_path    <- args[3]
output_path        <- args[4]
batch              <- as.numeric(args[5])
chr                <- as.numeric(args[6])

# Print the inputs so that the job log records exactly what was supplied.
cat("AGDS PATH:", agds_path, "\n")
cat("NULL MODEL PATH:", obj_nullmodel_path, "\n")
cat("ANNOTATION PATH:", annotation_path, "\n")
cat("OUTPUT PATH:", output_path, "\n")
cat("BATCH MAXIMUM:", batch, "\n")
cat("CHROMOSOME:", chr, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################################

## load packages
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)
library(STAAR)
library(STAARpipeline)

## Null model
obj_nullmodel <- get(load(obj_nullmodel_path))

## QC label
QC_label <- "annotation/info/QC_label"
## QC_label <- "annotation/filter" [Alternative]

## Variant type 
variant_type <- "SNV"
## Genotype missing imputaiton
geno_missing_imputation <- "mean"

## Annotation directory 
Annotation_dir <- "annotation/info/FunctionalAnnotation"
## Annotation channel
Annotation_name_catalog <- get(load(annotation_path))
## Use annotation weigths 
Use_annotation_weights <- TRUE
## Annotation name
Annotation_name <- c("CADD",
                     "LINSIGHT",
                     "FATHMM.XF",
                     "aPC.EpigeneticActive",
                     "aPC.EpigeneticRepressed",
                     "aPC.EpigeneticTranscription",
                     "aPC.Conservation",
                     "aPC.LocalDiversity",
                     "aPC.Mappability",
                     "aPC.TF",
                     "aPC.Protein")

########################################################### 
# CHROMOSOME / GENE SUBSETTING 
###########################################################

# IMPORTANT: 
# This line assumes that an object called `genes_info` already exists 
# in the R environment.
genes_info <- genes_info[genes_info[,2] == chr, , drop = FALSE]

subset <- c("DNMT3A", "TET2", "IDH2", "SRSF2")

genes_info <- dplyr::filter(
    genes_info,
    hgnc_symbol %in% subset
)

print(genes_info)

# Maximum number of genes processed in one batch. 
# Each batch will contain up to 50 genes. 
# NOTE: 
# This currently overrides the command-line `batch` argument.
gene_num_in_array <- 50
# gene_num_in_array <- batch [Alternative]

# Calculate the number of batches required to process all genes.
group.num.allchr <- ceiling(nrow(genes_info) / gene_num_in_array)

########################################################################### 
# BATCH LOOP
########################################################################### 

print(gene_num_in_array)
print(group.num.allchr)
print(genes_info)

# Process one batch at a time.
for (arrayid in 1:group.num.allchr) {

    # Current batch identifier.
    groupid <- arrayid
    # Total number of batches.
    group.num <- group.num.allchr

    cat("\n========================================\n")
    cat("BATCH INFORMATION\n")
    cat("========================================\n")
    cat("ARRAY ID:", arrayid, "\n")
    cat("CHROMOSOME:", chr, "\n")
    cat("GENES IN CHROMOSOME:", nrow(genes_info), "\n")
    cat("NUMBER OF BATCHES:", group.num, "\n")
    cat("========================================\n\n")

    # Copy the chromosome-specific gene table.
    genes_info_chr <- genes_info
    # Number of genes on the chromosome.
    sub_seq_num <- nrow(genes_info_chr)

    # Determine the row indices of genes belonging to this batch. 
    # For every batch except the last: 
    # batch 1 -> genes 1:50 
    # batch 2 -> genes 51:100 
    # batch 3 -> genes 101:150 
    # The final batch ends at the actual number of genes.
    if (groupid < group.num) {
        sub_seq_id <- ((groupid - 1) * gene_num_in_array + 1):
                      (groupid * gene_num_in_array)
    } else {
        sub_seq_id <- ((groupid - 1) * gene_num_in_array + 1):
                      sub_seq_num
    }

    # Open the chromosome-specific GDS file.
    genofile <- seqOpen(agds_path)
    # Print the path for debugging/logging.
    print(agds_path)

    # Every gene that causes an error is appended here.
    failed_genes <- character(0)

    # Process each gene assigned to this batch.
    for (kk in sub_seq_id) {

        # Retrieve the gene name from column 1 of genes_info_chr.
        gene_name <- genes_info_chr[kk, 1]

        print("\n========================================")
        print("GENE_CENTRIC_CODING INPUTS")
        print("========================================")

        print(paste("Chromosome:", chr))
        print(paste("Gene index:", kk))
        print(paste("Gene name:", gene_name))

        print("\n--- GDS ---")
        print(paste("AGDS path:", agds_path))
        print(genofile)

        print("\n--- NULL MODEL ---")
        print(paste("Null model path:", obj_nullmodel_path))
        print("Null model class:")
        print(class(obj_nullmodel))
        print("Null model structure:")
        print(str(obj_nullmodel, max.level = 2))

        print("\n--- TEST PARAMETERS ---")
        print(paste("rare_maf_cutoff:", 0.01))
        print(paste("rv_num_cutoff:", 0))
        print(paste("QC_label:", QC_label))
        print(paste("variant_type:", variant_type))
        print(paste("geno_missing_imputation:", geno_missing_imputation))

        print("\n--- ANNOTATION ---")
        print(paste("Annotation_dir:", Annotation_dir))
        print(paste("Annotation_path:", annotation_path))
        print(paste("Use_annotation_weights:", Use_annotation_weights))
        print("Annotation_name:")
        print(Annotation_name)

        print("\n--- ANNOTATION CATALOG ---")
        print("Class:")
        print(class(Annotation_name_catalog))
        print("Structure:")
        print(str(Annotation_name_catalog, max.level = 2))

        print("\n========================================")
        print("RUNNING Gene_Centric_Coding()")
        print("========================================")

        gds_ids <- seqGetData(genofile, "sample.id")
        null_ids <- obj_nullmodel$id_include
        
        cat("GDS samples:", length(gds_ids), "\n")
        cat("NULL model samples:", length(null_ids), "\n")
        
        cat("First GDS IDs:\n")
        print(head(gds_ids))
        
        cat("First NULL IDs:\n")
        print(head(null_ids))
        
        cat("Class GDS IDs:", class(gds_ids), "\n")
        cat("Class NULL IDs:", class(null_ids), "\n")
        
        cat("Exact intersection:\n")
        print(length(intersect(as.character(gds_ids), as.character(null_ids))))
        
        cat("GDS IDs matching NULL:\n")
        print(head(intersect(as.character(gds_ids), as.character(null_ids))))

        ################################################################### 
        # RUN GENE-CENTRIC TEST 
        ###################################################################


        results <- tryCatch({

            Gene_Centric_Coding(
                chr = chr,
                gene_name = gene_name,
                genofile = genofile,
                obj_nullmodel = obj_nullmodel,
                rare_maf_cutoff = 0.01,
                rv_num_cutoff = 0,
                QC_label = QC_label,
                variant_type = variant_type,
                geno_missing_imputation = geno_missing_imputation,
                Annotation_dir = Annotation_dir,
                Annotation_name_catalog = Annotation_name_catalog,
                Use_annotation_weights = Use_annotation_weights,
                Annotation_name = Annotation_name,
                SPA_p_filter = TRUE,
                p_filter_cutoff = 0.05
            )

        }, error = function(e) {

            cat("\n*** GENE SKIPPED ***\n")
            cat("Gene:", gene_name, "\n")
            cat("Chromosome:", chr, "\n")
            cat("Index:", kk, "\n")
            cat("Error:", conditionMessage(e), "\n")
            cat("*** CONTINUING TO NEXT GENE ***\n\n")

            failed_genes <<- c(
                failed_genes,
                paste0(
                    "chr", chr,
                    ":", gene_name,
                    ":", conditionMessage(e)
                )
            )

            return(NULL)
        })

        cat("\n========== FUNCTION RETURN ==========\n")
        cat("Gene:", gene_name, "\n")
        cat("is.null:", is.null(results), "\n")
        cat("class:", class(results), "\n")
        cat("length:", length(results), "\n")
        
        if (!is.null(results)) {
            str(results, max.level = 2)
        } else {
            cat("RESULT IS NULL\n")
        }
        
        cat("=====================================\n")

        ## Only store successful results
        if (!is.null(results)) {

            cat("\n========================================\n")
            cat("SUCCESS:", gene_name, "\n")
            cat("========================================\n")
        
            print(results)
        
            output_file <- file.path(
                output_path,
                paste0(
                    "STAAR_", gene_name,
                    "_gene_centric_coding_chr", chr,
                    "_arrayid_", arrayid,
                    ".Rdata"
                )
            )
        
            save(
                results,
                file = output_file
            )
        
            cat("SUCCESSFULLY SAVED:\n")
            cat(output_file, "\n")
        }
        
    }

    ####################################################################### 
    # CLOSE GDS 
    #######################################################################

    ## Close GDS after all genes in this batch
    seqClose(genofile)

    ####################################################################### 
    # BATCH SUMMARY 
    #######################################################################

    cat("\n========================================\n")
    cat("BATCH COMPLETE\n")
    cat("========================================\n")
    cat("ARRAY ID:", arrayid, "\n")
    cat("Failed/skipped genes:", length(failed_genes), "\n")
    cat("========================================\n\n")
}

Overwriting STAARpipeline_Gene_Centric_Coding.R


In [14]:
# Copy script in workspace bucket 
!gcloud storage cp STAARpipeline_Gene_Centric_Coding.R ${WORKSPACE_BUCKET}/data/scripts/

Copying file://STAARpipeline_Gene_Centric_Coding.R to gs://rw-migration-aou-rw-9f879fc4/data/scripts/STAARpipeline_Gene_Centric_Coding.R
  Completed files 1/1 | 10.7kiB/10.7kiB                                        


In [15]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/STAARpipeline_Gene_Centric_Coding.R 

#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Read command-line arguments supplied after: 
args <- commandArgs(trailingOnly = TRUE)

# Input arguments:
#   1. agds_path          = chromosome-specific annotated GDS file
#   2. obj_nullmodel_path = fitted STAAR/SAIGE null model RData
#   3. annotation_path   = annotation catalog RData
#   4. output_path        = directory for gene-level results
#   5. batch              = batch size / maximum genes per batch
#   6. chr                = chromosome to analyze
agds_path          <- args[1]
obj_nullmodel_path <- args[2]
annotation_path    <- args[3]
output_path        <- args[4]
batch              <- as.numeric(args[5])
chr                <- as.numeric(args[6])

# Print the inputs so that the job log records exactly what was supplied.
cat("AGDS PATH:", agds_path, "\n")
cat("NULL MODEL PATH:", obj_nullmode

# SUBMIT COMMAND 

In [16]:
import subprocess
import pandas as pd

# Create dataframe of size of exome plink files in GB 
chromosome = []
sizes = [] 

# Loop for all chromosome 
# for chrom in range(1, 22):
for chrom in [2, 4, 15, 17]:
    
    # Extract size of genomic plink bed file in GB for each chromosome 
    command = [
        "gcloud", "storage", "ls",
        "-l",
        f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/annotated_chr{chrom}.gds",
        f"--billing-project={os.environ['GOOGLE_PROJECT']}"
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)

    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

# Format genomic file size 
genomic_size['chromosome'] = genomic_size['chromosome'].astype('int')
genomic_size['size'] = (genomic_size['size'].astype('int') / 1e9) 

print(genomic_size)


   chromosome      size
0           2  1.975847
1           4  1.187610
2          15  1.047307
3          17  0.841535


In [17]:
import shlex

# Logging directory
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Dsub script
SCRIPT = "STAARpipeline_Gene_Centric_Coding.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

MAX_DISK_SIZE = 20 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 5 # Set additional disk size for each chromosome 

# Loop to submit dsub command
for CHROM in [2, 4, 15, 17]:

    print(f"Submitting dsub command for chromosome {CHROM}")

    DISK_SIZE = (
        round(
            float(genomic_size.loc[genomic_size["chromosome"] == CHROM, "size"
                ].iloc[0]
            )
        )
        + EXTRA_DISK_SIZE
    )

    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE:
        print(f"Disk too large for chromosome {CHROM}")
        break

    if DISK_SIZE > 1000:
        print(f"Disk too large for chromosome {CHROM}")
        break

    # File paths
    AGDS = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/annotated_chr{CHROM}.gds"
    ANNOTATION = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/preparation/Annotation_name_catalog_chr{CHROM}.Rdata"
    NULLMODEL = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/null_modle/obj_spa_nullmodel.Rdata"
    OUTPUT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/summary_statistics/aml_biosample_age_saddle_point"
    BATCH = 379
    
    # R script command
    R_COMMAND = 'Rscript "$SCRIPT" "$AGDS" "$NULLMODEL" "$ANNOTATION" "$OUTPUT" "$BATCH" "$CHROM"'
    
    command = [
        "aou_dsub",
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/zilinli/staarpipeline:0.9.7",
        "--boot-disk-size", "5",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-highmem-2",
        "--logging", LOGGING,
        "--input", f"SCRIPT={PATH}",
        "--input", f"AGDS={AGDS}",
        "--input", f"NULLMODEL={NULLMODEL}",
        "--input", f"ANNOTATION={ANNOTATION}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"BATCH={BATCH}",
        "--env", f"CHROM={CHROM}",
        "--command", R_COMMAND,
    ]
    
    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {shlex.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    
    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 2
Requesting disk size: 7
rscript--vienose--261107-204938-42

Job properties:
  job-id: rscript--vienose--261107-204938-42
  job-name: rscript
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/rscript--vienose--261107-204938-42-0-0
Launched job-id: rscript--vienose--261107-204938-42
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261107-204938-42' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261107-204938-42' --users 'vienose'

Submitting dsub command for chromosome 4
Requesting disk size: 6
rscript--vienose--261107-204941-72

Job properties:
  job-id: rscript--vienose--261107-204941-72
  job-name: rscript
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-64

In [10]:
!gcloud storage ls gs://rw-migration-aou-rw-9f879fc4/data/staar/summary_statistics/aml_biosample_age_saddle_point/

ERROR: (gcloud.storage.ls) One or more URLs matched no objects.


In [24]:
!gcloud storage cp gs://rw-migration-aou-rw-9f879fc4/data/staar/summary_statistics/aml_biosample_age_saddle_point/* .

Copying gs://rw-migration-aou-rw-9f879fc4/data/staar/summary_statistics/aml_biosample_age_saddle_point/STAAR_DNMT3A_gene_centric_coding_chr2_arrayid_1.Rdata to file://./STAAR_DNMT3A_gene_centric_coding_chr2_arrayid_1.Rdata
Copying gs://rw-migration-aou-rw-9f879fc4/data/staar/summary_statistics/aml_biosample_age_saddle_point/STAAR_IDH2_gene_centric_coding_chr15_arrayid_1.Rdata to file://./STAAR_IDH2_gene_centric_coding_chr15_arrayid_1.Rdata
Copying gs://rw-migration-aou-rw-9f879fc4/data/staar/summary_statistics/aml_biosample_age_saddle_point/STAAR_SRSF2_gene_centric_coding_chr17_arrayid_1.Rdata to file://./STAAR_SRSF2_gene_centric_coding_chr17_arrayid_1.Rdata
⠛ Completed files 0 | 0B                                                       Copying gs://rw-migration-aou-rw-9f879fc4/data/staar/summary_statistics/aml_biosample_age_saddle_point/STAAR_TET2_gene_centric_coding_chr4_arrayid_1.Rdata to file://./STAAR_TET2_gene_centric_coding_chr4_arrayid_1.Rdata
  Completed files 4/4 | 7.3kiB/7.3k